In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from IPython.core.pylabtools import figsize
DB_PATH = "/Users/arnarfreyrerlingsson/Desktop/Polymarket_bitcoin_5min/db/polymarket.duckdb"

In [2]:
def query_to_df(query, limit=1000):
    HARD_LIMIT = 5_000_000
    if limit:
        limit = f"\n LIMIT {limit}"
    else:
        limit = f""
        #limit = f"\n LIMIT {HARD_LIMIT}"

    con = duckdb.connect(DB_PATH,read_only=True)
    try:
        query = str(query) + limit
        df = con.execute(query).df()
        return df
    finally:
        con.close()

In [3]:
query = "SHOW ALL TABLES"
query_to_df(query,None)

,database,schema,name,column_names,column_types,temporary
0,polymarket,main,MARKET_TRADES,"[THE_TIME, MARKET, PRICE, SIZE, SIDE, OUTCOME,...","[TIMESTAMP WITH TIME ZONE, VARCHAR, DOUBLE, DO...",False
1,polymarket,main,bitcoin_prices,"[OPEN_TIME, CLOSE_TIME, OPEN, HIGH, LOW, CLOSE...","[TIMESTAMP WITH TIME ZONE, TIMESTAMP WITH TIME...",False
2,polymarket,main,btc_history,"[SLUG, TOKEN_ID, OUTCOME, TIMESTAMP, PRICE]","[VARCHAR, VARCHAR, VARCHAR, TIMESTAMP WITH TIM...",False
3,polymarket,main,btc_summary,"[SLUG, TITLE, CONDITION_ID, UP_TOKEN_ID, DOWN_...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False
4,polymarket,main,btc_trades,"[SLUG, CONDITION_ID, EVENT_SLUG, TITLE, ICON, ...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False


### Market summary table

In [4]:
query = f"""

WITH MARKETS_SUMMARY AS (
SELECT
    SlUG AS MARKET,
    MARKET_ID,
    EVENT_STARTTIME AS EVENT_START,
    EVENT_END_DATE AS EVENT_CLOSE,
    PRICE_TO_BEAT,
    FINAL_PRICE,
    RESULT,VOLUME,
    N_TRADES,
    EVENT_START_DATE AS EVENT_OPEN

FROM btc_summary
WHERE PRICE_TO_BEAT IS NOT NULL
)
SELECT *
FROM MARKETS_SUMMARY
"""

#query_to_df(query)

### Market trades raw table

In [5]:
query = f"""
WITH MARKET_TRADES AS (
SELECT SlUG AS MARKET,TIMESTAMP AS THE_TIME,SECS_INTO_WINDOW,SIDE,OUTCOME,PRICE,SIZE,WALLET,NAME AS USER_NAME,
PSEUDONYM AS USER_PSUEDO
FROM btc_trades)

SELECT *
FROM MARKET_TRADES
ORDER BY THE_TIME ASC
"""
#query_to_df(query)


### Market trades main table

In [6]:
MARKET = 'btc-updown-5m-1780272000'

query = f"""
SELECT *
FROM MARKET_TRADES
"""

query_to_df(query,1000)

,THE_TIME,MARKET,PRICE,SIZE,SIDE,OUTCOME,PRICE_TO_BEAT,FINAL_PRICE,RESULT,VOLUME,N_TRADES,SECS_INTO_WINDOW,EVENT_OPEN,EVENT_START,EVENT_CLOSE,WALLET,USER_NAME,USER_PSUEDO
0,2026-05-31 23:50:46+00:00,btc-updown-5m-1780272000,0.50,10.400000,BUY,Down,73574.662155,73758.025335,Up,49500.575071,1900,-554,2026-05-31 00:18:50.912679+00:00,2026-06-01 00:00:00+00:00,2026-06-01 00:05:00+00:00,0x0c4e9cc4f8987607d51fb761eb0c2696f442bf78,0x0C4e9cC4f8987607D51fb761Eb0c2696f442bF78-177...,Curvy-Laborer
1,2026-05-31 23:51:05+00:00,btc-updown-5m-1780272000,0.50,10.400000,BUY,Down,73574.662155,73758.025335,Up,49500.575071,1900,-535,2026-05-31 00:18:50.912679+00:00,2026-06-01 00:00:00+00:00,2026-06-01 00:05:00+00:00,0x0c4e9cc4f8987607d51fb761eb0c2696f442bf78,0x0C4e9cC4f8987607D51fb761Eb0c2696f442bF78-177...,Curvy-Laborer
2,2026-05-31 23:55:03+00:00,btc-updown-5m-1780272000,0.50,12.500000,BUY,Down,73574.662155,73758.025335,Up,49500.575071,1900,-297,2026-05-31 00:18:50.912679+00:00,2026-06-01 00:00:00+00:00,2026-06-01 00:05:00+00:00,0xced5a61d4e8c0e85df6d28351bba2d367664cd53,oyjhahaha,Damp-Recommendation
3,2026-05-31 23:55:05+00:00,btc-updown-5m-1780272000,0.50,5.000000,BUY,Down,73574.662155,73758.025335,Up,49500.575071,1900,-295,2026-05-31 00:18:50.912679+00:00,2026-06-01 00:00:00+00:00,2026-06-01 00:05:00+00:00,0x7ee1b149f81124ad566422b25bf23b8dab00a7f1,AdamOfEternia,Variable-Innovation
4,2026-05-31 23:55:07+00:00,btc-updown-5m-1780272000,0.51,9.800000,BUY,Up,73574.662155,73758.025335,Up,49500.575071,1900,-293,2026-05-31 00:18:50.912679+00:00,2026-06-01 00:00:00+00:00,2026-06-01 00:05:00+00:00,0xa7ab95b9b75c905efbf6e1e65748dc4305c83734,mazala-,Humongous-Force
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2026-06-01 00:02:01+00:00,btc-updown-5m-1780272000,0.59,8.474575,BUY,Down,73574.662155,73758.025335,Up,49500.575071,1900,121,2026-05-31 00:18:50.912679+00:00,2026-06-01 00:00:00+00:00,2026-06-01 00:05:00+00:00,0x3295bd10cd5ceabcf626d1d484c7ed2200ed6f72,dpcruz,Superficial-Travel
996,2026-06-01 00:02:01+00:00,btc-updown-5m-1780272000,0.58,10.517240,BUY,Down,73574.662155,73758.025335,Up,49500.575071,1900,121,2026-05-31 00:18:50.912679+00:00,2026-06-01 00:00:00+00:00,2026-06-01 00:05:00+00:00,0xaffc3335c7a9679696adb348679264b87406a372,hello-cli,Spotted-Forest
997,2026-06-01 00:02:03+00:00,btc-updown-5m-1780272000,0.63,20.000000,BUY,Down,73574.662155,73758.025335,Up,49500.575071,1900,123,2026-05-31 00:18:50.912679+00:00,2026-06-01 00:00:00+00:00,2026-06-01 00:05:00+00:00,0x251c1a283703beed41590b0875a8dcb8ddd1541f,0xAAAAA,Suburban-Characterization
998,2026-06-01 00:02:03+00:00,btc-updown-5m-1780272000,0.64,4.330000,BUY,Down,73574.662155,73758.025335,Up,49500.575071,1900,123,2026-05-31 00:18:50.912679+00:00,2026-06-01 00:00:00+00:00,2026-06-01 00:05:00+00:00,0xfb0f17657c9c24293b918adb86362a4d8fc90b02,aoe2gamer,Nice-Prejudice


In [13]:

SECS_RANGE = [55,60] # first min
PRICE_RANGE = [0.7,0.8]
outcome = 'Up'
side = 'BUY'

query = f"""
SELECT MARKET,OUTCOME,PRICE,SECS_INTO_WINDOW,SIZE,SIDE,PRICE_TO_BEAT,FINAL_PRICE,RESULT
FROM MARKET_TRADES
WHERE SECS_INTO_WINDOW BETWEEN {SECS_RANGE[0]} AND {SECS_RANGE[1]}
AND PRICE BETWEEN {PRICE_RANGE[0]} AND {PRICE_RANGE[1]}
AND OUTCOME = '{outcome}'
AND SIDE = '{side}'
"""

query_to_df(query,1000)

,MARKET,OUTCOME,PRICE,SECS_INTO_WINDOW,SIZE,SIDE,PRICE_TO_BEAT,FINAL_PRICE,RESULT
0,btc-updown-5m-1780272600,Up,0.700000,60,14.000000,BUY,73647.273446,73694.864786,Up
1,btc-updown-5m-1780272600,Up,0.700000,60,5.000000,BUY,73647.273446,73694.864786,Up
2,btc-updown-5m-1780273500,Up,0.720000,55,5.000000,BUY,73697.764179,73851.617310,Up
3,btc-updown-5m-1780273500,Up,0.720000,55,5.138885,BUY,73697.764179,73851.617310,Up
4,btc-updown-5m-1780273500,Up,0.720000,55,5.000000,BUY,73697.764179,73851.617310,Up
...,...,...,...,...,...,...,...,...,...
995,btc-updown-5m-1780357200,Up,0.740000,57,5.405403,BUY,71417.122340,71418.992802,Up
996,btc-updown-5m-1780357200,Up,0.730000,57,5.000000,BUY,71417.122340,71418.992802,Up
997,btc-updown-5m-1780357200,Up,0.728037,57,62.000000,BUY,71417.122340,71418.992802,Up
998,btc-updown-5m-1780357200,Up,0.709457,57,80.000000,BUY,71417.122340,71418.992802,Up


In [28]:
market = 'btc-updown-5m-1781094600'

query_to_df(f"""
        WITH MARKET_TRADES AS (SELECT SlUG AS MARKET,
        TIMESTAMP AS THE_TIME,SECS_INTO_WINDOW,SIDE,OUTCOME,PRICE,
        SIZE,WALLET,NAME AS USER_NAME,PSEUDONYM AS USER_PSUEDO
        FROM btc_trades),
        BASE AS (SELECT * FROM MARKET_TRADES
        WHERE MARKET = '{market}'
        AND SECS_INTO_WINDOW >= 30
        AND SECS_INTO_WINDOW <= 301
        ORDER BY THE_TIME ASC)
        SELECT * FROM BASE""",10000)



,MARKET,THE_TIME,SECS_INTO_WINDOW,SIDE,OUTCOME,PRICE,SIZE,WALLET,USER_NAME,USER_PSUEDO
0,btc-updown-5m-1781094600,2026-06-10 12:30:30+00:00,30,SELL,Up,0.95,76.0,0x5e2b9261b0c4f697b55bf921ff2bc227183d9101,,
1,btc-updown-5m-1781094600,2026-06-10 12:30:30+00:00,30,SELL,Up,0.95,76.0,0x75cc3b63a2f2423085e10706c78b494017b93ce1,,
2,btc-updown-5m-1781094600,2026-06-10 12:30:30+00:00,30,BUY,Down,0.04,37.1,0x3654e2c92f08f722e8983db1f5d48c8962bf1b78,tenx006,Identical-Childhood
3,btc-updown-5m-1781094600,2026-06-10 12:30:30+00:00,30,BUY,Up,0.97,32.0,0x3654e2c92f08f722e8983db1f5d48c8962bf1b78,tenx006,Identical-Childhood
4,btc-updown-5m-1781094600,2026-06-10 12:30:30+00:00,30,BUY,Down,0.04,25.0,0xb4e586443a3464cd8a05940529cd06180f89b608,0xB4e586443a3464cD8a05940529cD06180F89b608-177...,Glistening-Enigma
...,...,...,...,...,...,...,...,...,...,...
1190,btc-updown-5m-1781094600,2026-06-10 12:34:06+00:00,246,BUY,Down,0.01,25.0,0x1d99336f5c7dabec1b890cb2237f9b4aa139b04c,traxeast,Grubby-Fee
1191,btc-updown-5m-1781094600,2026-06-10 12:34:15+00:00,255,BUY,Down,0.01,467.0,0xe9076a87c5ed90ef16e6fe6529c943baeca0cff6,suntori,All-Peasant-Drug
1192,btc-updown-5m-1781094600,2026-06-10 12:34:21+00:00,261,BUY,Down,0.01,100.0,0x3f3842e83d34378f0ef445e9b3eedd0bcd1ae0d7,patherking,Inborn-Indicator
1193,btc-updown-5m-1781094600,2026-06-10 12:34:42+00:00,282,BUY,Down,0.01,25.0,0x1d99336f5c7dabec1b890cb2237f9b4aa139b04c,traxeast,Grubby-Fee
